# Cross-Phase Statistical and Consistency Audit

Run this notebook **after Phases 0-5 have been rerun with the final-revision notebooks**.

It creates a single traceable results registry, deduplicates repeated cross-phase comparisons, applies Holm-Bonferroni correction across prespecified inferential families, and checks that significance-table MAEs agree with the final ensemble metrics. Phase 5 is labeled exploratory and kept separate from the main Phase 0-4 inferential families.


In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Crypto_Research')
ROOT = PROJECT_DIR / 'revised_outputs_v4'
OUT = ROOT / 'final_revision_audit'
OUT.mkdir(parents=True, exist_ok=True)
PHASE_DIRS = {
    'phase0': ROOT / 'phase0',
    'phase1': ROOT / 'phase1',
    'phase2': ROOT / 'phase2',
    'phase3': ROOT / 'phase3',
    'phase4': ROOT / 'phase4',
    'phase5': ROOT / 'phase5_reliability_fusion',
}


Mounted at /content/drive


In [2]:

# FINAL REVISION: one deterministic implementation of Holm-Bonferroni for the cross-phase family audit.
def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)
    idx = np.where(np.isfinite(values))[0]
    if len(idx) == 0:
        return out
    p = values[idx]
    order = np.argsort(p)
    p_sorted = p[order]
    m = len(p_sorted)
    adj_sorted = np.empty(m)
    running = 0.0
    for rank, value in enumerate(p_sorted):
        running = max(running, min(1.0, (m-rank) * float(value)))
        adj_sorted[rank] = running
    unsorted = np.empty(m)
    unsorted[order] = adj_sorted
    out[idx] = unsorted
    return out

metric_frames = []
sig_frames = []
for phase, directory in PHASE_DIRS.items():
    metric_path = directory / 'ensemble_metrics.csv'
    sig_path = directory / 'significance_final.csv'
    if metric_path.exists():
        frame = pd.read_csv(metric_path)
        frame.insert(0, 'source_phase', phase)
        frame['source_file'] = str(metric_path)
        metric_frames.append(frame)
    if sig_path.exists():
        frame = pd.read_csv(sig_path)
        frame.insert(0, 'source_phase', phase)
        frame['source_file'] = str(sig_path)
        sig_frames.append(frame)

if not metric_frames or not sig_frames:
    raise FileNotFoundError('Final phase outputs are incomplete. Rerun Phases 0-5 before the consistency audit.')

metrics = pd.concat(metric_frames, ignore_index=True, sort=False)
significance = pd.concat(sig_frames, ignore_index=True, sort=False)


In [3]:

# FINAL REVISION: build a unique metric registry so manuscript numbers are copied from one source of truth.
metric_cols = ['source_phase','asset','phase','model','method','mae','rmse','r2','directional_accuracy','directional_accuracy_p_value','observation_count','source_file']
metric_cols = [c for c in metric_cols if c in metrics.columns]
registry = metrics[metric_cols].copy()
registry['result_id'] = (
    registry['source_phase'].astype(str) + '__' + registry['asset'].astype(str) + '__' + registry['method'].astype(str)
)
registry = registry.sort_values(['source_phase','asset','method']).drop_duplicates('result_id', keep='last')
if registry['result_id'].duplicated().any():
    raise AssertionError('Duplicate result IDs remain in the final registry.')
registry.to_csv(OUT / 'paper1_results_registry.csv', index=False)
display(registry)


,source_phase,asset,phase,model,method,mae,rmse,r2,directional_accuracy,directional_accuracy_p_value,observation_count,source_file,result_id
0,phase0,BTC,benchmark,arima,arima,0.016677,0.024011,-0.001876,0.483553,0.735911,304,/content/drive/MyDrive/Crypto_Research/revised...,phase0__BTC__arima
2,phase0,BTC,phase0,lstm,phase0_lstm,0.016727,0.024127,-0.011607,0.483553,0.735911,304,/content/drive/MyDrive/Crypto_Research/revised...,phase0__BTC__phase0_lstm
3,phase0,BTC,phase0,xgb,phase0_xgb,0.017511,0.024908,-0.078127,0.496711,0.568287,304,/content/drive/MyDrive/Crypto_Research/revised...,phase0__BTC__phase0_xgb
1,phase0,BTC,benchmark,random_walk_zero_return,random_walk_zero_return,0.016665,0.024013,-0.002082,NaN,NaN,304,/content/drive/MyDrive/Crypto_Research/revised...,phase0__BTC__random_walk_zero_return
4,phase0,ETH,benchmark,arima,arima,0.022992,0.033659,-0.005683,0.519737,0.264089,304,/content/drive/MyDrive/Crypto_Research/revised...,phase0__ETH__arima
...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,phase5,ETH,phase5_concat,lstm,phase5_concat_lstm,0.023891,0.034016,-0.027106,0.486842,0.697101,304,/content/drive/MyDrive/Crypto_Research/revised...,phase5__ETH__phase5_concat_lstm
89,phase5,ETH,phase5_concat,xgb,phase5_concat_xgb,0.024039,0.034301,-0.044407,0.516447,0.302899,304,/content/drive/MyDrive/Crypto_Research/revised...,phase5__ETH__phase5_concat_xgb
90,phase5,ETH,phase5_reliability_fusion,lstm,phase5_reliability_fusion_lstm,0.023150,0.033834,-0.016128,0.490132,0.655932,304,/content/drive/MyDrive/Crypto_Research/revised...,phase5__ETH__phase5_reliability_fusion_lstm
91,phase5,ETH,phase5_reliability_fusion,xgb,phase5_reliability_fusion_xgb,0.024521,0.034647,-0.065579,0.509868,0.387172,304,/content/drive/MyDrive/Crypto_Research/revised...,phase5__ETH__phase5_reliability_fusion_xgb


In [4]:
# FINAL REVISION: repeated method outputs across phase notebooks must agree numerically.
repeat_rows = []
for (asset, method), group in metrics.groupby(['asset', 'method']):
    if len(group) <= 1:
        continue
    for metric in ['mae', 'rmse', 'r2', 'directional_accuracy']:
        if metric not in group.columns:
            continue
        values = pd.to_numeric(group[metric], errors='coerce').dropna()
        if len(values) == 0:
            continue
        repeat_rows.append({
            'asset': asset, 'method': method, 'metric': metric,
            'occurrences': int(len(values)), 'min_value': float(values.min()),
            'max_value': float(values.max()), 'range': float(values.max()-values.min()),
            'pass_1e_10': bool((values.max()-values.min()) <= 1e-10),
        })
repeated_method_consistency = pd.DataFrame(repeat_rows)
repeated_method_consistency.to_csv(OUT / 'repeated_method_consistency_check.csv', index=False)
display(repeated_method_consistency)
if not repeated_method_consistency.empty and not repeated_method_consistency['pass_1e_10'].all():
    raise AssertionError('Repeated method metrics differ across phase notebooks after canonical reuse. Inspect repeated_method_consistency_check.csv before manuscript drafting.')


CANONICAL_ALIASES = [
    ('phase0', 'phase0', 'phase1', 'phase0'),
    ('phase0', 'phase0', 'phase2', 'phase0'),
    ('phase0', 'phase0', 'phase3', 'phase0'),
    ('phase2', 'phase2', 'phase3', 'phase2_general'),
    ('phase2', 'phase2', 'phase5', 'phase2_general'),
    ('phase3', 'phase3_crypto', 'phase4', 'phase3_reference'),
    ('phase3', 'phase3_crypto', 'phase5', 'phase3_crypto'),
]

def _compare_prediction_alias(source_file_name, source_phase_dir, source_phase_key, target_phase_dir, target_phase_key):
    source_path = PHASE_DIRS[source_phase_dir] / source_file_name
    target_path = PHASE_DIRS[target_phase_dir] / source_file_name
    rows = []
    if not source_path.exists() or not target_path.exists():
        return rows
    source = pd.read_csv(source_path)
    target = pd.read_csv(target_path)
    for model in ['lstm','xgb']:
        sm = f'{source_phase_key}_{model}'
        tm = f'{target_phase_key}_{model}'
        left = source[(source['phase'].astype(str)==source_phase_key) & (source['method'].astype(str)==sm)].copy()
        right = target[(target['phase'].astype(str)==target_phase_key) & (target['method'].astype(str)==tm)].copy()
        keys = ['asset','seed','target_date']
        if 'walk_forward_fold' in left.columns and 'walk_forward_fold' in right.columns:
            keys.append('walk_forward_fold')
        cols = keys + ['actual_return','predicted_return']
        left = left[cols].rename(columns={'actual_return':'actual_source','predicted_return':'pred_source'})
        right = right[cols].rename(columns={'actual_return':'actual_target','predicted_return':'pred_target'})
        merged = left.merge(right, on=keys, how='outer', indicator=True)
        complete = merged['_merge'].eq('both')
        actual_diff = np.abs(merged.loc[complete,'actual_source'] - merged.loc[complete,'actual_target'])
        pred_diff = np.abs(merged.loc[complete,'pred_source'] - merged.loc[complete,'pred_target'])
        passed = bool(
            len(merged) > 0
            and complete.all()
            and (actual_diff <= 1e-12).all()
            and (pred_diff <= 1e-12).all()
        )
        rows.append({
            'file': source_file_name,
            'source_phase_dir': source_phase_dir,
            'source_method': sm,
            'target_phase_dir': target_phase_dir,
            'target_method': tm,
            'rows_compared': int(complete.sum()),
            'max_actual_abs_diff': float(actual_diff.max()) if len(actual_diff) else np.nan,
            'max_prediction_abs_diff': float(pred_diff.max()) if len(pred_diff) else np.nan,
            'pass_1e_12': passed,
        })
    return rows

alias_rows=[]
for src_dir, src_key, tgt_dir, tgt_key in CANONICAL_ALIASES:
    alias_rows.extend(_compare_prediction_alias('predictions.csv', src_dir, src_key, tgt_dir, tgt_key))
    alias_rows.extend(_compare_prediction_alias('walk_forward_predictions.csv', src_dir, src_key, tgt_dir, tgt_key))
canonical_reuse = pd.DataFrame(alias_rows)
canonical_reuse.to_csv(OUT / 'canonical_prediction_reuse_check.csv', index=False)
display(canonical_reuse)
if canonical_reuse.empty or not canonical_reuse['pass_1e_12'].all():
    raise AssertionError('Canonical prediction reuse check failed. Do not draft the manuscript until all alias checks pass.')


,asset,method,metric,occurrences,min_value,max_value,range,pass_1e_10
0,BTC,arima,mae,6,0.016677,0.016677,0.000000e+00,True
1,BTC,arima,rmse,6,0.024011,0.024011,0.000000e+00,True
2,BTC,arima,r2,6,-0.001876,-0.001876,0.000000e+00,True
3,BTC,arima,directional_accuracy,6,0.483553,0.483553,0.000000e+00,True
4,BTC,phase0_lstm,mae,4,0.016727,0.016727,1.006140e-16,True
...,...,...,...,...,...,...,...,...
73,ETH,phase3_crypto_xgb,r2,2,-0.089797,-0.089797,4.996004e-16,True
74,ETH,phase3_crypto_xgb,directional_accuracy,2,0.500000,0.500000,0.000000e+00,True
75,ETH,random_walk_zero_return,mae,6,0.022939,0.022939,0.000000e+00,True
76,ETH,random_walk_zero_return,rmse,6,0.033596,0.033596,0.000000e+00,True


,file,source_phase_dir,source_method,target_phase_dir,target_method,rows_compared,max_actual_abs_diff,max_prediction_abs_diff,pass_1e_12
0,predictions.csv,phase0,phase0_lstm,phase1,phase0_lstm,3040,0.0,0.000000e+00,True
1,predictions.csv,phase0,phase0_xgb,phase1,phase0_xgb,3040,0.0,0.000000e+00,True
2,walk_forward_predictions.csv,phase0,phase0_lstm,phase1,phase0_lstm,3040,0.0,3.388132e-21,True
3,walk_forward_predictions.csv,phase0,phase0_xgb,phase1,phase0_xgb,3040,0.0,0.000000e+00,True
4,predictions.csv,phase0,phase0_lstm,phase2,phase0_lstm,3040,0.0,0.000000e+00,True
5,predictions.csv,phase0,phase0_xgb,phase2,phase0_xgb,3040,0.0,0.000000e+00,True
6,walk_forward_predictions.csv,phase0,phase0_lstm,phase2,phase0_lstm,3040,0.0,3.388132e-21,True
7,walk_forward_predictions.csv,phase0,phase0_xgb,phase2,phase0_xgb,3040,0.0,0.000000e+00,True
8,predictions.csv,phase0,phase0_lstm,phase3,phase0_lstm,3040,0.0,0.000000e+00,True
9,predictions.csv,phase0,phase0_xgb,phase3,phase0_xgb,3040,0.0,0.000000e+00,True


In [5]:

# FINAL REVISION: remove comparisons repeated in later notebooks, then correct multiplicity across each scientific family.
key = ['asset','candidate_method','reference_method']
sig = significance.sort_values('source_phase').drop_duplicates(key, keep='first').reset_index(drop=True)

sig['candidate_result_id'] = sig['source_phase'].astype(str) + '__' + sig['asset'].astype(str) + '__' + sig['candidate_method'].astype(str)
sig['reference_result_id'] = sig['source_phase'].astype(str) + '__' + sig['asset'].astype(str) + '__' + sig['reference_method'].astype(str)
registry_ids = set(registry['result_id'].astype(str))
missing_ids = sorted((set(sig['candidate_result_id']) | set(sig['reference_result_id'])) - registry_ids)
if missing_ids:
    raise AssertionError(f'Significance registry references missing result IDs: {missing_ids[:10]}')


# Phase 5 is exploratory by design and is never pooled into a Phase 0-4 primary family.
sig['final_family'] = sig['comparison_family'].astype(str)
sig.loc[sig['source_phase'].eq('phase5'), 'final_family'] = 'phase5_exploratory'

sig['dm_p_value_holm_cross_phase_family'] = np.nan
sig['wilcoxon_p_value_holm_cross_phase_family'] = np.nan
for family, idx in sig.groupby('final_family').groups.items():
    idx = list(idx)
    sig.loc[idx, 'dm_p_value_holm_cross_phase_family'] = holm_adjust(sig.loc[idx, 'dm_p_value_two_sided'])
    sig.loc[idx, 'wilcoxon_p_value_holm_cross_phase_family'] = holm_adjust(sig.loc[idx, 'wilcoxon_p_value_two_sided'])

sig.to_csv(OUT / 'paper1_significance_registry.csv', index=False)
display(sig.sort_values(['final_family','asset','comparison']))


,source_phase,asset,comparison_family,comparison,candidate_method,reference_method,candidate_mae,reference_mae,mae_difference,mean_loss_differential,...,wilcoxon_p_value_candidate_lower_error,paired_observations,dm_p_value_holm_within_family,wilcoxon_p_value_holm_within_family,source_file,candidate_result_id,reference_result_id,final_family,dm_p_value_holm_cross_phase_family,wilcoxon_p_value_holm_cross_phase_family
51,phase3,BTC,benchmark_comparisons,Crypto FinBERT LSTM vs ARIMA,phase3_crypto_lstm,arima,0.017040,0.016677,3.630943e-04,3.630943e-04,...,0.990943,304,0.136545,0.108679,/content/drive/MyDrive/Crypto_Research/revised...,phase3__BTC__phase3_crypto_lstm,phase3__BTC__arima,benchmark_comparisons,0.409634,0.362263
53,phase3,BTC,benchmark_comparisons,Crypto FinBERT LSTM vs random walk,phase3_crypto_lstm,random_walk_zero_return,0.017040,0.016665,3.752861e-04,3.752861e-04,...,0.996031,304,0.076509,0.079375,/content/drive/MyDrive/Crypto_Research/revised...,phase3__BTC__phase3_crypto_lstm,phase3__BTC__random_walk_zero_return,benchmark_comparisons,0.229526,0.238126
46,phase3,BTC,benchmark_comparisons,Crypto FinBERT XGB vs ARIMA,phase3_crypto_xgb,arima,0.017683,0.016677,1.005550e-03,1.005550e-03,...,0.994660,304,0.189181,0.085437,/content/drive/MyDrive/Crypto_Research/revised...,phase3__BTC__phase3_crypto_xgb,phase3__BTC__arima,benchmark_comparisons,0.546523,0.277669
48,phase3,BTC,benchmark_comparisons,Crypto FinBERT XGB vs random walk,phase3_crypto_xgb,random_walk_zero_return,0.017683,0.016665,1.017742e-03,1.017742e-03,...,0.995899,304,0.189181,0.079375,/content/drive/MyDrive/Crypto_Research/revised...,phase3__BTC__phase3_crypto_xgb,phase3__BTC__random_walk_zero_return,benchmark_comparisons,0.546523,0.238126
52,phase3,BTC,benchmark_comparisons,General FinBERT LSTM vs ARIMA,phase2_general_lstm,arima,0.016615,0.016677,-6.222478e-05,-6.222478e-05,...,0.147244,304,0.906136,0.588977,/content/drive/MyDrive/Crypto_Research/revised...,phase3__BTC__phase2_general_lstm,phase3__BTC__arima,benchmark_comparisons,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,phase4,ETH,temporal_ablation,Combined vs short window only for XGB,phase4_combined_xgb,phase4_short_window_only_xgb,0.024258,0.024258,0.000000e+00,0.000000e+00,...,1.000000,304,1.000000,1.000000,/content/drive/MyDrive/Crypto_Research/revised...,phase4__ETH__phase4_combined_xgb,phase4__ETH__phase4_short_window_only_xgb,temporal_ablation,1.000000,1.000000
57,phase4,ETH,temporal_ablation,Decay only vs Phase 3 for LSTM,phase4_decay_only_lstm,phase3_reference_lstm,0.023302,0.023513,-2.111632e-04,-2.111632e-04,...,0.064871,304,1.000000,1.000000,/content/drive/MyDrive/Crypto_Research/revised...,phase4__ETH__phase4_decay_only_lstm,phase4__ETH__phase3_reference_lstm,temporal_ablation,1.000000,1.000000
63,phase4,ETH,temporal_ablation,Decay only vs Phase 3 for XGB,phase4_decay_only_xgb,phase3_reference_xgb,0.024686,0.024686,1.387779e-17,1.321550e-17,...,1.000000,304,1.000000,1.000000,/content/drive/MyDrive/Crypto_Research/revised...,phase4__ETH__phase4_decay_only_xgb,phase4__ETH__phase3_reference_xgb,temporal_ablation,1.000000,1.000000
56,phase4,ETH,temporal_ablation,Short window only vs Phase 3 for LSTM,phase4_short_window_only_lstm,phase3_reference_lstm,0.022989,0.023513,-5.236587e-04,-5.236587e-04,...,0.013214,304,0.319112,0.422842,/content/drive/MyDrive/Crypto_Research/revised...,phase4__ETH__phase4_short_window_only_lstm,phase4__ETH__phase3_reference_lstm,temporal_ablation,0.319112,0.422842


In [6]:

# CONSISTENCY FIX: numerical traceability uses exact source-phase result IDs, never cross-phase method averages.
metric_lookup = registry.set_index('result_id')['mae'].to_dict()
checks = []
for _, row in sig.iterrows():
    candidate_registry = metric_lookup.get(row['candidate_result_id'], np.nan)
    reference_registry = metric_lookup.get(row['reference_result_id'], np.nan)
    checks.append({
        'source_phase': row['source_phase'],
        'asset': row['asset'], 'comparison': row['comparison'],
        'candidate_method': row['candidate_method'], 'reference_method': row['reference_method'],
        'candidate_result_id': row['candidate_result_id'], 'reference_result_id': row['reference_result_id'],
        'candidate_mae_significance': row['candidate_mae'], 'candidate_mae_registry': candidate_registry,
        'reference_mae_significance': row['reference_mae'], 'reference_mae_registry': reference_registry,
        'candidate_abs_diff': abs(row['candidate_mae']-candidate_registry) if np.isfinite(candidate_registry) else np.nan,
        'reference_abs_diff': abs(row['reference_mae']-reference_registry) if np.isfinite(reference_registry) else np.nan,
    })
consistency = pd.DataFrame(checks)
consistency['pass_1e_10'] = (
    consistency['candidate_abs_diff'].notna()
    & consistency['reference_abs_diff'].notna()
    & (consistency['candidate_abs_diff'] <= 1e-10)
    & (consistency['reference_abs_diff'] <= 1e-10)
)
consistency.to_csv(OUT / 'metric_significance_consistency_check.csv', index=False)
display(consistency)
if not consistency['pass_1e_10'].all():
    raise AssertionError('Significance-to-registry traceability failed. Inspect metric_significance_consistency_check.csv.')
print(f'PASS: {int(consistency["pass_1e_10"].sum())}/{len(consistency)} significance-to-registry checks.')


,source_phase,asset,comparison,candidate_method,reference_method,candidate_result_id,reference_result_id,candidate_mae_significance,candidate_mae_registry,reference_mae_significance,reference_mae_registry,candidate_abs_diff,reference_abs_diff,pass_1e_10
0,phase0,BTC,Phase 0 LSTM vs random walk,phase0_lstm,random_walk_zero_return,phase0__BTC__phase0_lstm,phase0__BTC__random_walk_zero_return,0.016727,0.016727,0.016665,0.016665,0.0,0.0,True
1,phase0,BTC,Phase 0 LSTM vs ARIMA,phase0_lstm,arima,phase0__BTC__phase0_lstm,phase0__BTC__arima,0.016727,0.016727,0.016677,0.016677,0.0,0.0,True
2,phase0,BTC,Phase 0 XGB vs random walk,phase0_xgb,random_walk_zero_return,phase0__BTC__phase0_xgb,phase0__BTC__random_walk_zero_return,0.017511,0.017511,0.016665,0.016665,0.0,0.0,True
3,phase0,BTC,Phase 0 XGB vs ARIMA,phase0_xgb,arima,phase0__BTC__phase0_xgb,phase0__BTC__arima,0.017511,0.017511,0.016677,0.016677,0.0,0.0,True
4,phase0,ETH,Phase 0 LSTM vs random walk,phase0_lstm,random_walk_zero_return,phase0__ETH__phase0_lstm,phase0__ETH__random_walk_zero_return,0.023778,0.023778,0.022939,0.022939,0.0,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,phase5,BTC,Dual-scope concat vs crypto for LSTM,phase5_concat_lstm,phase3_crypto_lstm,phase5__BTC__phase5_concat_lstm,phase5__BTC__phase3_crypto_lstm,0.016971,0.016971,0.017040,0.017040,0.0,0.0,True
108,phase5,BTC,Dual-scope concat vs general for LSTM,phase5_concat_lstm,phase2_general_lstm,phase5__BTC__phase5_concat_lstm,phase5__BTC__phase2_general_lstm,0.016971,0.016971,0.016615,0.016615,0.0,0.0,True
109,phase5,ETH,Reliability fusion XGB vs random walk,phase5_reliability_fusion_xgb,random_walk_zero_return,phase5__ETH__phase5_reliability_fusion_xgb,phase5__ETH__random_walk_zero_return,0.024521,0.024521,0.022939,0.022939,0.0,0.0,True
110,phase5,BTC,Reliability fusion XGB vs random walk,phase5_reliability_fusion_xgb,random_walk_zero_return,phase5__BTC__phase5_reliability_fusion_xgb,phase5__BTC__random_walk_zero_return,0.017416,0.017416,0.016665,0.016665,0.0,0.0,True


PASS: 112/112 significance-to-registry checks.


In [7]:

# Collect revision-specific evidence files without inventing results when a phase has not yet been rerun.
evidence_files = {
    'temporal_alignment': OUT / 'temporal_alignment_protocol.json',
    'timestamp_audit': OUT / 'timestamp_convention_audit.csv',
    'phase4_decay_selection': PHASE_DIRS['phase4'] / 'decay_rate_selection_summary.csv',
    'phase4_decay_lock': PHASE_DIRS['phase4'] / 'selected_decay_rate.json',
    'phase5_reliability_sensitivity': PHASE_DIRS['phase5'] / 'reliability_coverage_sensitivity.csv',
    'phase5_provider_shift': PHASE_DIRS['phase5'] / 'provider_shift_weight_diagnostics.csv',
}
evidence_status = pd.DataFrame([
    {'item': k, 'path': str(v), 'exists': v.exists()} for k,v in evidence_files.items()
])
evidence_status.to_csv(OUT / 'final_revision_evidence_status.csv', index=False)
display(evidence_status)

checklist = """# Paper 1 final consistency checklist

- Require `canonical_prediction_reuse_check.csv`, `repeated_method_consistency_check.csv`, and `metric_significance_consistency_check.csv` to pass before manuscript drafting.
- Copy all reported MAE/RMSE/R2/DA values from `paper1_results_registry.csv`.
- Copy inferential p-values from `paper1_significance_registry.csv`; identify raw vs Holm-adjusted values explicitly.
- Use DM-HAC as the primary forecast-loss comparison and two-sided Wilcoxon as the nonparametric robustness check.
- Do not compare classification accuracy values from prior literature numerically with this study's regression MAE/RMSE/R2.
- State the UTC news window, forecast origin, and close-to-close target interval explicitly.
- Report the validation-selected Phase 4 decay half-life/rate exactly as stored in `selected_decay_rate.json`.
- Use five-seed walk-forward summaries, not the legacy seed-42-only result.
- Keep Phase 5 exploratory and brief; report the provider-volume sensitivity audit.
- Regenerate final figures from final CSV outputs; do not transcribe intermediate console numbers.
"""
(OUT / 'manuscript_consistency_checklist.md').write_text(checklist, encoding='utf-8')
print(checklist)


,item,path,exists
0,temporal_alignment,/content/drive/MyDrive/Crypto_Research/revised...,True
1,timestamp_audit,/content/drive/MyDrive/Crypto_Research/revised...,True
2,phase4_decay_selection,/content/drive/MyDrive/Crypto_Research/revised...,True
3,phase4_decay_lock,/content/drive/MyDrive/Crypto_Research/revised...,True
4,phase5_reliability_sensitivity,/content/drive/MyDrive/Crypto_Research/revised...,True
5,phase5_provider_shift,/content/drive/MyDrive/Crypto_Research/revised...,True


# Paper 1 final consistency checklist

- Require `canonical_prediction_reuse_check.csv`, `repeated_method_consistency_check.csv`, and `metric_significance_consistency_check.csv` to pass before manuscript drafting.
- Copy all reported MAE/RMSE/R2/DA values from `paper1_results_registry.csv`.
- Copy inferential p-values from `paper1_significance_registry.csv`; identify raw vs Holm-adjusted values explicitly.
- Use DM-HAC as the primary forecast-loss comparison and two-sided Wilcoxon as the nonparametric robustness check.
- Do not compare classification accuracy values from prior literature numerically with this study's regression MAE/RMSE/R2.
- State the UTC news window, forecast origin, and close-to-close target interval explicitly.
- Report the validation-selected Phase 4 decay half-life/rate exactly as stored in `selected_decay_rate.json`.
- Use five-seed walk-forward summaries, not the legacy seed-42-only result.
- Keep Phase 5 exploratory and brief; report the provider-volume sensit